### Experimental Setup
 - Pick three simulations as described in the paper: Linear, Nonlinear, Random

 - Train on each individual simulation, use a simple linear probe to reconstruct past tokens and see reconstruction error. This will show the hardness of each memorization task

 - Next train on one of the simulations completely, then see whether the autoencoder can remember the other two simulations without being trained on it.

In [26]:
## Load necessary library files ##

import sys
sys.path.append('..')
from source.utils import get_sequence
from source.model.memory import Memory

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch.nn.functional as F
import random 
import pickle 

In [3]:
## select device ##
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")  # works only with NVIDIA GPUs (not on Mac)
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [9]:
def linear_sequence(total_samples, token_number=7):
    sequence = ""
    for i in range(total_samples):
        sequence += (chr((i % token_number) + ord('A')))
    return sequence

In [23]:
def random_sequence(total_samples, token_number=7):
    sequence = ""
    for i in range(total_samples):
        sequence += chr(np.random.randint(token_number, size=1)[0] + ord('A'))
    return sequence

## Visualize the simulation data ##

In [28]:
print("A 42 tokens long linear sequence ", linear_sequence(42))

## load the nonlinear simulation from source files ##
print("A 42 tokens long nonlinear sequence ", get_sequence(42, n_community=2, n_members=3))

print("A 42 tokens long random sequence ", random_sequence(42))

A 42 tokens long linear sequence  ABCDEFGABCDEFGABCDEFGABCDEFGABCDEFGABCDEFG
A 42 tokens long nonlinear sequence  ABCGDEFGDEFGACBGDFEGEDFGDFEGEFDGEFDGDEFGDE
A 42 tokens long random sequence  FABEDADFGACDABDFDDGABBBACEDGDCECAAEFFGAEBE


In [27]:
## define memory parameters ##
input_size = 7
hidden_size = 50
embedding_dim = 30

In [29]:
## define a linear probe to reconstruct memory ##

class linear_probe(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.linear = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x):
        out = self.linear(x)

        return out 

In [ ]:
reps = 10
short_term_memories = [2,4,6,8,10]
past_recalls = [1,2,3,4,5]
total_samples = 10000
lr = 6e-4
repitition = []
acc = []
test_acc = []
test_acc_decoder = []
bptt = []
recalls = []


for rep in range(reps):
    for short_term_memory in short_term_memories:
        model = Memory(input_size, embedding_dim, hidden_size)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-8)
        criterion = path_finder_loss

        data = get_random_sequence(total_samples, token_number=vocab_size-1)

        data_set = Dataset_converter(data, working_memory, short_term_memory)
        train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

        total = 0
        correct = np.zeros(1000,dtype=float)
        decoder_correct = np.zeros(1000,dtype=float)
        for X, y in train_loader:
            decoder_input = torch.cat([torch.zeros((1, 1), dtype=torch.long), X[:, :-1]], dim=1)
            decoder_target = X.flip(1)

            optimizer.zero_grad()

            if total == 0:
                y_pred, decoder_output, h = model(X, decoder_input)
            else:
                y_pred, decoder_output, h = model(X, decoder_input, hidden)

            loss = criterion(y_pred[0], decoder_output[0], y[0], decoder_target[0])     
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            with torch.no_grad():
                hidden = h.detach()
                total += 1

                if y[0] == y_pred[0].argmax():
                        correct[total%1000] = 1
                else:
                    correct[total%1000] = 0

                if torch.sum(decoder_output.argmax(2) == decoder_target)==short_term_memory:
                        decoder_correct[total%1000] = 1
                else:
                    decoder_correct[total%1000] = 0

                test_acc.append(
                    np.sum(correct)/total if total<1000 else np.sum(correct)/1000
                )
                test_acc_decoder.append(
                    np.sum(decoder_correct)/total if total<1000 else np.sum(decoder_correct)/1000
                )
                if total%1000 == 0:
                    print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}, decoder accuracy: {test_acc_decoder[-1]:.4f}')



        ### extract the hidden states from the trained RNN ###
        hidden_states = []

        for X, y in train_loader:
            with torch.no_grad():
                if total == 0:
                    y_pred, h = model.encoder(X)
                else:
                    y_pred, h = model.encoder(X, h)

                hidden_states.append(
                    h[0][0]
                )

        ### train classifier to reconstruct past tokens ###
        for past_recall in past_recalls:
            print('Doing recall ', past_recall)
            data_set = Dataset_reconstructer(hidden_states, data, short_term_memory=short_term_memory, past_recall=past_recall)
            reconstruct_loader = DataLoader(data_set, batch_size=1, shuffle=False)

            reconstructor = classifier(vocab_size, hidden_size)
            optimizer = torch.optim.SGD(reconstructor.parameters(), lr=lr, momentum=0.95)
            criterion = torch.nn.CrossEntropyLoss()

            total = 0
            # correct = np.zeros(1000,dtype=float)
            print('Training reconstruction classifier ...')
            for epoch in tqdm(range(10)):
                for X, y in reconstruct_loader:
                    optimizer.zero_grad()

                    y_pred = reconstructor(X)
                    

                    loss = criterion(y_pred, y)     
                    loss.backward()
                    optimizer.step()

            print("Evaluating reconstruction ...")
            correct = 0
            for X, y in reconstruct_loader:
                with torch.no_grad():
                    y_pred = reconstructor(X)
                    
                    if y_pred.argmax() == y:
                        correct += 1

            print('Totoal accuracy :', correct/len(data_set))
            repitition.append(rep)
            acc.append(correct/len(data_set))
            bptt.append(short_term_memory)
            recalls.append(past_recall)
            
            
            
df = pd.DataFrame()
df['reps'] = repitition
df['accuracy'] = acc 
df['BBPTT'] = bptt 
df['Recall'] = recalls

with open('../pickle_files/memory_capacity_random_autoencoder.pickle', 'wb') as f:
    pickle.dump(df, f)